# GLM-5.3-Flash 企业部署：8×H20 / FP8 / vLLM

按顺序执行，每个代码单元前说明用途、输出和使用 GPU 的时点。操作目标为**专用 Ubuntu 24.04 x86_64 服务器**，不是共享训练服务器。模型为 `zai-org/GLM-5.3-Flash`；API 名为 `glm-5.3-flash`。

- `/srv/phyxi` 必须是管理员预先挂载的数据卷；本 Notebook 不格式化磁盘。
- 推荐主机内存 ≥768 GiB（脚本门槛 600 GiB）、数据卷 ≥1.5 TB，首次制备至少 600 GiB 空闲。单份 FP8 权重约 306 GiB，实际文件量在下载后记录。
- 安装驱动可能要求重启，企业 CA 签发需要管理员提交 CSR。这两步有明确停点，完成后继续，不要忽略错误。
- 完成准备后，只有“启动服务”单元会初始化 GPU 并加载模型；本地静态验证不代表目标服务器已验收。
- 初始上下文 64K、调度序列上限 16 是可调整部署参数，**不是实测承载人数**。
- 图片请求使用 Python 前端；H20 使用 FP8 权重与 BF16 KV，非 FP8 KV。不开启 MTP/PD 分离，先完成基础服务验收。

运行前先按 README 安装并启动 Jupyter；此 Notebook 的管理员 kernel 仅监听回环，通过 SSH 隧道访问。
本文路径以默认 `/srv/phyxi` 展示；修改 `data_root` 后，生成的目录、挂载、脚本和 systemd 路径会同步替换。


## 0. 定位仓库和运行环境

本单元不修改系统。将内核目录定位到仓库，后续动作由 `scripts/deploy.py` 的具名阶段执行；所有命令在脚本中可审阅。遇到任何命令失败即停止，不继续后续阶段。
`stage("docker")` 这类调用只是具名步骤入口，等价于 `python3 scripts/deploy.py docker --config config.local.json`。参数来自第 1 节，无需打开脚本修改。


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
REPO = Path.cwd().resolve()
if REPO.name == "notebooks": REPO = REPO.parent
assert (REPO / "scripts/deploy.py").is_file(), "请在本仓库内打开 Notebook"
assert os.geteuid() == 0, "需要按 README 启动管理员 Notebook 内核"
CONFIG_PATH = REPO / "config.local.json"  # 已加入 .gitignore；不要提交执行后的 Notebook

def stage(name):
    subprocess.run(["/usr/bin/python3", str(REPO / "scripts/deploy.py"), name,
                    "--config", str(CONFIG_PATH)], check=True)
print("Repository:", REPO)
print((REPO / "deployment/versions.json").read_text())

## 1. 集中修改与确认部署配置

**在下面的 `OVERRIDES` 单元格直接修改，不需要编辑 `.py`、`.sh`、JSON 或 YAML 文件。**

- `None` 表示保留上次保存值；首次运行时使用下表默认值。
- 要改某项，将 `None` 换成实际值：字符串加引号，数字不加引号，多个值用列表。例如 `"domain": "llm.company.internal"`。
- 首次必须填写域名、IP、允许网段，并确认专用服务器和数据卷。GPU UUID 可在驱动安装后的第 5 节获取，再回到这里填写。
- 执行本单元后写入被 Git 忽略的 `config.local.json`；`stage(...)` 每次都重新读取它。修改 Notebook 但未执行此单元，不会生效。
- 在第 10 节首次生成正式配置之前完成版本及资源参数的确定。已生成配置后改动属于发布变更，须按第 10 节说明处理，不直接覆盖正在运行的服务。

| 字段 | 是否需要修改 | 默认值 | 获取方式 / 修改影响 |
|---|---|---|---|
| `dedicated_server` | 必须确认 | `true` | 确认是专用服务器；共享训练服务器不使用此 Notebook |
| `domain` | 必须填写 | `示例域名不可用` | 向企业网络管理员获取 DNS 域名，解析到 intranet_ip |
| `intranet_ip` | 必须填写 | `10.20.0.10 为示例` | 用 ip -br addr 查看服务器业务网卡地址 |
| `allowed_cidrs` | 必须填写 | `示例网段` | 由网络管理员给出工作站/应用出口网段；不是 GPU 网段 |
| `gpu_uuids` | 必须填写 | `空列表` | 第 5 节 nvidia-smi 获取八张已分配 H20 的完整 UUID；驱动未装时可暂留空 |
| `max_model_len` | 可调整 | `65536` | 单序列输入+输出上限；提高会增加缓存需求 |
| `max_num_seqs` | 可调整 | `16` | 服务调度上限，非企业用户数；按实测延迟调优 |
| `gpu_memory_utilization` | 可调整 | `0.85` | 每卡显存预算比例，不是权重精度 |
| `release` | 必须确认 | `20261108-01` | 本次发布编号；首次配置生成后不覆盖已有变更版本 |
| `tls_mode` | 必须选择 | `enterprise` | 正式用 enterprise；封闭测试可选 self-signed |
| `tls_chain_source` | 企业证书必填 | `/secure-transfer/server-chain.pem` | 企业 CA 返回的服务器完整证书链在本机的路径 |
| `tls_ca_source` | 企业证书必填 | `/secure-transfer/company-ca.pem` | 工作站信任用企业 CA 文件在本机的路径 |
| `alertmanager` | 有告警平台时填写 | `空字符串` | 填主机:端口；留空只采集指标，不向企业通知平台发送告警 |
| `docker_version` | 可保留 | `空字符串` | 首次从官方源选 29.x 并记录；企业有批准版本时填 apt 完整版本字符串 |
| `toolkit_version` | 可保留 | `1.20.1-1` | 填写 APT 完整版本，非只有 1.20.1 |
| `data_root` | 必须确认 | `/srv/phyxi` | 填写已挂载数据卷路径，使用 findmnt 检查；不填已有业务目录 |
| `driver_branch` | 可保留 | `580` | 已有驱动须不低于该分支；安装新驱动需要重启 |
| `model_revision` | 可保留 | `仓库锁定 commit` | 只填写官方 GLM-5.3-Flash 的 40 位 commit；变更后重新下载与验收 |
| `vllm_image` | 可保留 | `仓库锁定 digest` | 填写完整镜像名@sha256:摘要；不可用裸 nightly 代替 |
| `nginx_image` | 可保留 | `nginx:stable-alpine` | 首次解析 digest 固定；企业可填批准的完整 digest |
| `prometheus_image` | 可保留 | `prom/prometheus:v3.5.0` | 首次解析 digest 固定；企业可填批准的完整 digest |
| `model_cpus` | 可调整 | `32` | 模型容器 CPU 配额，按专用主机可用 CPU 确定 |
| `model_memory_gib` | 可调整 | `512` | 模型容器主机内存上限；主机至少多留 64 GiB |
| `shm_gib` | 可调整 | `32` | 容器共享内存预算 |
| `max_num_batched_tokens` | 可调整 | `4096` | 单次调度输入与生成 token 总预算 |
| `max_images` | 可调整 | `4` | 每请求最大图片数；当前不开放视频 |
| `reasoning_effort` | 可选择 | `low` | 只能选 low / high / max；均可能输出思考 token |
| `per_client_connections` | 可调整 | `4` | 网关单工作站 Token 在途连接限制 |
| `total_connections` | 可调整 | `16` | 网关全局在途限制，应不小于单工作站限制 |
| `benchmark_concurrency` | 可调整 | `[1, 2, 4]` | 客户端并发档位；超过单 Token 限制时可能返回 429 |
| `benchmark_requests` | 可调整 | `16` | 每个并发档位的请求总数，预热请求另计 |
| `benchmark_max_tokens` | 可调整 | `1024` | 并发测试每请求输出上限，包含思考 token |
| `verify_max_tokens` | 可调整 | `4096` | 功能验收输出上限；预算太小可能只有思考没有答案 |

**无需自行编造的值：** 后端 Key、工作站 Token 由第 10 节生成；CSR 和 TLS 私钥由第 11 节生成。域名和网段由企业网络管理员确定，证书由企业 CA 签发，不能用随机值代替。

**本方案固定项：** 官方 GLM-5.3-Flash FP8、8 张 H20 / TP8、Python 多模态前端、BF16 KV、glm47 解析器、API 名 `glm-5.3-flash`。这些是方案兼容性基线，不是需要你编辑的脚本参数。


In [ ]:
# 读取已有配置，补齐本版本新字段；未填写的字段采用示例默认值。
config = json.loads((REPO / "deployment/config.example.json").read_text())
if CONFIG_PATH.exists():
    config.update(json.loads(CONFIG_PATH.read_text()))

# 直接在本单元把需要修改的 None 替换成实际值。None = 保留当前值。
OVERRIDES = {
    "dedicated_server": None,  # 必须确认：确认是专用服务器；共享训练服务器不使用此 Notebook
    "domain": None,  # 必须填写：向企业网络管理员获取 DNS 域名，解析到 intranet_ip
    "intranet_ip": None,  # 必须填写：用 ip -br addr 查看服务器业务网卡地址
    "allowed_cidrs": None,  # 必须填写：由网络管理员给出工作站/应用出口网段；不是 GPU 网段
    "gpu_uuids": None,  # 必须填写：第 5 节 nvidia-smi 获取八张已分配 H20 的完整 UUID；驱动未装时可暂留空
    "max_model_len": None,  # 可调整：单序列输入+输出上限；提高会增加缓存需求
    "max_num_seqs": None,  # 可调整：服务调度上限，非企业用户数；按实测延迟调优
    "gpu_memory_utilization": None,  # 可调整：每卡显存预算比例，不是权重精度
    "release": None,  # 必须确认：本次发布编号；首次配置生成后不覆盖已有变更版本
    "tls_mode": None,  # 必须选择：正式用 enterprise；封闭测试可选 self-signed
    "tls_chain_source": None,  # 企业证书必填：企业 CA 返回的服务器完整证书链在本机的路径
    "tls_ca_source": None,  # 企业证书必填：工作站信任用企业 CA 文件在本机的路径
    "alertmanager": None,  # 有告警平台时填写：填主机:端口；留空只采集指标，不向企业通知平台发送告警
    "docker_version": None,  # 可保留：首次从官方源选 29.x 并记录；企业有批准版本时填 apt 完整版本字符串
    "toolkit_version": None,  # 可保留：填写 APT 完整版本，非只有 1.20.1
    "data_root": None,  # 必须确认：填写已挂载数据卷路径，使用 findmnt 检查；不填已有业务目录
    "driver_branch": None,  # 可保留：已有驱动须不低于该分支；安装新驱动需要重启
    "model_revision": None,  # 可保留：只填写官方 GLM-5.3-Flash 的 40 位 commit；变更后重新下载与验收
    "vllm_image": None,  # 可保留：填写完整镜像名@sha256:摘要；不可用裸 nightly 代替
    "nginx_image": None,  # 可保留：首次解析 digest 固定；企业可填批准的完整 digest
    "prometheus_image": None,  # 可保留：首次解析 digest 固定；企业可填批准的完整 digest
    "model_cpus": None,  # 可调整：模型容器 CPU 配额，按专用主机可用 CPU 确定
    "model_memory_gib": None,  # 可调整：模型容器主机内存上限；主机至少多留 64 GiB
    "shm_gib": None,  # 可调整：容器共享内存预算
    "max_num_batched_tokens": None,  # 可调整：单次调度输入与生成 token 总预算
    "max_images": None,  # 可调整：每请求最大图片数；当前不开放视频
    "reasoning_effort": None,  # 可选择：只能选 low / high / max；均可能输出思考 token
    "per_client_connections": None,  # 可调整：网关单工作站 Token 在途连接限制
    "total_connections": None,  # 可调整：网关全局在途限制，应不小于单工作站限制
    "benchmark_concurrency": None,  # 可调整：客户端并发档位；超过单 Token 限制时可能返回 429
    "benchmark_requests": None,  # 可调整：每个并发档位的请求总数，预热请求另计
    "benchmark_max_tokens": None,  # 可调整：并发测试每请求输出上限，包含思考 token
    "verify_max_tokens": None,  # 可调整：功能验收输出上限；预算太小可能只有思考没有答案
}
config.update({key: value for key, value in OVERRIDES.items() if value is not None})
CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2) + "\n")
CONFIG_PATH.chmod(0o600)
DATA_ROOT = Path(config["data_root"])
print("以下为已保存配置（无密钥），请核对后再执行安装：")
print(json.dumps(config, ensure_ascii=False, indent=2))


## 2. 服务器与数据卷检查

检查 Ubuntu 24.04、x86_64、主机内存、独立挂载点与可用空间；拒绝已存在其他容器或 GPU 工作负载的服务器。输出硬件信息。未分配数据卷时先由管理员完成挂载，再重跑本步。
**执行前确定：** `data_root` 的挂载点、磁盘容量和 `domain` 已填妥。可在管理员终端使用 `findmnt`、`lsblk -f`、`df -h`、`ip -br addr` 查看磁盘和网卡。Notebook 不选择磁盘或自动格式化。


In [ ]:
stage("preflight")

## 3. 系统基础依赖与目录

安装证书、网络、Python、内核头文件等组件；创建 `/srv/phyxi/{docker,containerd,models,cache,prometheus,secrets,artifacts,evidence,releases}`，启用 SSH 和时间同步。

In [ ]:
stage("base")

## 4. NVIDIA 驱动

复用已正常工作的 580 或更新驱动；若缺失，安装官方 580 分支驱动。本步安装后主动抛出“需要重启”提示：在管理员终端执行 `reboot`，重启 Jupyter，重新运行 0、1、4 单元。Secure Boot 按提示完成 MOK 注册。不要在 Notebook 中自动重启。
**可在第 1 节修改：** `driver_branch`。默认 580，已有更高且正常工作的驱动直接复用；若企业选其他分支，应先确认它与锁定镜像的 CUDA 兼容。


In [ ]:
stage("driver")

## 5. 记录 GPU UUID

读取 GPU 清单，不加载模型。将下方输出的八张 H20 UUID 填入第 1 节 `gpu_uuids`，重新执行第 1 节保存。企业方案使用一份模型跨八卡 TP=8。
**需要填写：** `gpu_uuids`。复制下面的八个完整 UUID 到第 1 节列表，不能填 `[0,1,2,3,4,5,6,7]`。配置值只定义设备选择，启动前还会检查它们是否空闲。


In [ ]:
subprocess.run(["nvidia-smi", "--query-gpu=index,uuid,name,memory.total,memory.used", "--format=csv"], check=True)
assert len(json.loads(CONFIG_PATH.read_text())["gpu_uuids"]) == 8, "先更新第 1 节，填入八张卡的 UUID"

## 6. 安装 Docker 与持久存储

从 Docker 官方仓库选择 29.x，或使用第 1 节指定的批准版本。首次配置镜像与容器存储到 `/srv/phyxi/docker`，containerd 到 `/srv/phyxi/containerd`。记录版本；重跑核对已管理的 Engine，不搬迁已有业务数据。此阶段会重启专用服务器的 Docker。
**可在第 1 节修改：** `docker_version`、`data_root`。空版本表示首次自动选择官方源的 Docker 29.x 完整版本，并保存结果。已有批准版本则填完整 APT 字符串。设置过 Engine 存储后不直接修改数据根；数据迁移是独立运维操作。


In [ ]:
stage("docker")

## 7. 安装 GPU 容器运行时

安装固定版本 NVIDIA Container Toolkit，注册 NVIDIA runtime，记录版本并锁定软件包。重启专用服务器 Docker；不加载模型。
**可在第 1 节修改：** `toolkit_version`，例如 `1.20.1-1`。本步已经安装并记录后不会自动升级；变更版本需要维护窗口。


In [ ]:
stage("toolkit")

## 8. 固定镜像与 GLM 兼容性检查

拉取 `deployment/versions.json` 中固定 digest 的 vLLM 镜像；Nginx/Prometheus 首次解析 digest 后保存，后续复用。运行容器检查 GLM5Next 架构、glm47 工具/思考解析器、FlashInfer 最低版本和启动参数；不分配 GPU。结果在 `artifacts/image-compatibility.json`。失败时停止，不能仅凭镜像标签继续。
**可在第 1 节修改：** `vllm_image`、`nginx_image`、`prometheus_image`。默认直接使用仓库批准的输入，不要求你自行寻找新版本。首次镜像锁定后重复执行复用摘要；版本变更按发布流程处理，不能通过重跑单元悄悄升级。


In [ ]:
stage("images")

## 9. 下载固定版本 FP8 权重

下载官方仓库指定 commit；已有同一版本时断点续传。核对 FP8 配置和全部权重分片，逐文件生成 SHA-256。模型目录、总字节数、分片数写入 `artifacts/model.json`。此步磁盘与网络开销较大，不占 GPU。
**可在第 1 节修改：** `model_revision`；默认固定官方 commit 即可。无需修改 `download-model.py`。镜像支持和模型 commit 应配套验收。


In [ ]:
stage("model")

## 10. 生成正式配置与两层密钥

根据第 1 节生成 Compose 的八卡 UUID 列表、模型参数、Nginx 域名与允许网段。建立 `current` 发布软链接；生成后端 API Key、工作站 Token 及网关映射，不打印秘密。配置位于 `releases/<release>/`，凭据位于 `secrets/`；重复运行保留原有凭据。
**本步将配置落到文件：** GPU、资源预算和运行参数 → Compose / `.env`；域名、网段、连接限制 → Nginx；告警地址 → Prometheus。可以在第 13 节查看生成结果。首次生成后若更改配置，脚本会阻止覆盖已有发布；发布变更按仓库运维流程准备新目录，不在正在运行的目录中手改文件。


In [ ]:
stage("render")

## 11. 生成 TLS 私钥与 CSR

生成服务器私钥和证书签发请求。企业 CA 模式：将 `/srv/phyxi/artifacts/server.csr` 提交 CA；收到完整证书链和根/中间 CA 信任文件后，放到第 1 节配置的 `/secure-transfer/` 路径。私钥始终留在服务器。
**执行前确定：** 第 1 节 `domain` 是最终域名，`tls_mode` 是正式或测试模式。企业证书须使用本步骤生成的 CSR 签发；自行使用另一个私钥签发的证书无法与这里的私钥配对。


In [ ]:
stage("tls_csr")

## 12. 安装并验证 TLS 证书

校验证书域名、链和私钥配对；设置 Nginx 读取权限。企业 CA 文件未就绪则停止；自签名验收模式会生成 30 天证书，并把该证书用作工作站信任锚。
**直接在第 1 节填文件路径：** `tls_chain_source` 与 `tls_ca_source`。复制 CA 文件到服务器后再执行；若为 self-signed，这两个输入路径不读取。私钥和后端 Key 不手工填写到 Notebook。


In [ ]:
stage("tls_install")

## 13. 检查将要执行的配置

本单元只查看配置。关键基线：TP=8，FP8 权重自动按 checkpoint 读取；`dtype=bfloat16` 为未量化计算与 KV 默认精度；`VLLM_USE_RUST_FRONTEND=0`；两种解析器均为 glm47。不是 Qwen 参数。

默认 `MAX_NUM_SEQS=16` 为模型调度初值；Nginx 单工作站在途 4、总在途 16。多个工作站须分别发放 Token。参数解释见 `docs/parameters.md`。
**在第 1 节调节：** 上下文、序列数、显存比例、批 token 预算、图片数、思考强度、CPU/内存/shared memory、网关连接上限。下面显示的是已经生成的发布配置；改完输入需在允许的发布阶段重新生成，不会直接改变已运行模型。


In [ ]:
print((DATA_ROOT / "current/start-model.sh").read_text())
print((DATA_ROOT / "current/.env").read_text())  # 不包含密钥
subprocess.run(["bash", str(DATA_ROOT / "current/dc.sh"), "config", "--quiet"], check=True)


## 14. 启动服务（本步开始使用八张 GPU）

先核对八卡空闲和 DNS，再运行轻量 CUDA 检查、Nginx 配置检查，安装并启动 `phyxi-model.service`。启动最长等待约 65 分钟，用于首次权重加载和内核制备；网关在模型 healthy 后启动。

失败时在管理员终端查看 `bash /srv/phyxi/current/dc.sh logs --tail 100 model`。不要反复绕过检查，先修复日志中的具体错误。初次安装保持批准的软件源网络可达，完成启动验收后再执行企业出口策略。
**执行前确认：** 本步骤会占用八张 GPU；DNS、证书、入口网络策略和第 13 节参数均已确认。没有额外脚本参数需要修改。


In [ ]:
stage("start")

## 15. TLS、鉴权、图片及多轮工具协议验收

检查错误凭据返回 401、合法模型列表、文本、SSE、红色色块图片、工具调用与第二轮工具回填。默认使用 GLM 的 `reasoning_effort=low`，最多 4,096 输出 token（含思考）；不使用 Qwen 的 enable_thinking 开关。保存 `evidence/api-validation.json`。检查 Prometheus 就绪。
**可在第 1 节修改：** `verify_max_tokens`、`reasoning_effort`。网关 URL、CA、Token 路径和模型名由部署配置自动传入。


In [ ]:
stage("verify")

## 16. 有界并发验收

默认按客户端并发 1、2、4，分别预热一次并执行 16 个单轮请求，最多 1,024 输出 token。完成后保存每请求状态、实际 token、耗时、截断数与可见回答数。TPS 是服务合计 completion_tokens / 测试窗口，包含思考 token、网络与排队；不是单用户文字速度。结果为本机实测，不引用历史 27B 数据。

此组为协议/短负载验收；长期吞吐与多轮建模用户容量需另用项目实际任务集测试。
**可在第 1 节修改：** `benchmark_concurrency`、`benchmark_requests`、`benchmark_max_tokens`、`reasoning_effort`。这些仅影响验收请求，默认分别为 `[1,2,4]`、16、1024、low；不需要编辑负载脚本。


In [ ]:
stage("benchmark")

## 17. 导出工作站配置

生成仅管理员可读的 `/srv/phyxi/client-export/`，包含 CA、该工作站 Token、启动脚本和网关地址。不包含后端 Key、TLS 私钥。由企业安全渠道分发给指定工作站；新增工作站需发放不同 Token。

Windows 安装和项目接入执行仓库 `scripts/setup-workstation.ps1`，详见 `docs/workstation.md`。服务端 Notebook 不能代替在 Windows 上安装客户端和 CATIA。项目对话链路已提供版本校验的连接适配；实际 CATIA 工具链是独立业务验收。
**工作站需要确定：** 服务端导出的 endpoint.json 给出 BaseUrl；将 CA、Token 安全分发后，在 Windows 命令中填写这两个本地文件路径。不要把服务器路径直接当作 Windows 路径。


In [ ]:
stage("export")

## 18. 状态与交付记录

确认模型、网关、监控状态，保存软件版本与 GPU 采样。根据 `docs/operations.md` 完成告警送达、证书续签、主机重启恢复和回滚演练，再签署企业交付验收。
**可在第 1 节填写：** `alertmanager`。填写前向企业运维确认主机、端口和通知通道；空值不会发送告警通知。CPU/内存和初始并发阈值由验收数据确认。


In [ ]:
stage("status")
packages = subprocess.check_output(["dpkg-query", "-W", "-f=${Package}\t${Version}\n", "docker-ce", "docker-ce-cli", "containerd.io", "docker-compose-plugin", "nvidia-container-toolkit"], text=True)
(DATA_ROOT / "artifacts/system-packages.tsv").write_text(packages)
print("配置、版本和结果位于：", DATA_ROOT)


## 19. 可选：停止服务并释放 GPU

默认不停止正式服务。需要结束验收时将下面 `STOP_SERVICE` 改为 True。停止 systemd 管理的模型、网关和监控，保留权重、配置、缓存和证据。重启正式服务使用 `systemctl start phyxi-model.service`；启动前检查分配的 GPU 空闲。

In [ ]:
STOP_SERVICE = False
if STOP_SERVICE:
    stage("stop")
else:
    print("未执行停止操作")